In [1]:
import pandas as pd 

test = pd.read_csv('space/raw/test.csv')


def missing(df, feature):
    total = df[feature].isnull().sum()
    percent = (total / len(df)) * 100
    stats = pd.DataFrame({'total': total, 'percentage': percent})
    stats = stats[stats['total'] > 0]
    return stats 

In [16]:
numerical_test = test.select_dtypes(include=['int64', 'float64']).columns
categorical_test = test.select_dtypes(include='object').columns

num_test = missing(test, numerical_test)
print('for nuemrical test: ')
print(num_test)


cat_test = missing(test, categorical_test)
print('\n for categorical test: ')
print(cat_test)

for nuemrical test: 
Empty DataFrame
Columns: [total, percentage]
Index: []

 for categorical test: 
Empty DataFrame
Columns: [total, percentage]
Index: []


In [3]:
test['Age'] = test['Age'].fillna(test['Age'].median())

test['RoomService'] = test['RoomService'].fillna(test['RoomService'].median())

test['FoodCourt'] = test['FoodCourt'].fillna(test['FoodCourt'].median())

test['ShoppingMall'] = test['ShoppingMall'].fillna(test['ShoppingMall'].median())

test['Spa'] = test['Spa'].fillna(test['Spa'].median())

test['VRDeck'] = test['VRDeck'].fillna(test['VRDeck'].median())

In [5]:
test['GroupId'] = test['PassengerId'].str.split('_').str[0]
group_homeplanet = test.groupby('GroupId')['HomePlanet'].agg(lambda x: x.mode()[0] if len(x.mode()) > 0 else None)

test['HomePlanet'] = test.apply(
    lambda row: group_homeplanet[row['GroupId']] if pd.isnull(row['HomePlanet']) else row['HomePlanet'], 
    axis = 1
)

In [6]:
test['CabinDeck'] = test['Cabin'].str[0]

In [10]:
missing_homeplanet_with_deck = test[test['HomePlanet'].isnull() & test['CabinDeck'].notnull()]
deck_stats_missing = missing_homeplanet_with_deck['CabinDeck'].value_counts()
print(deck_stats_missing)

missing_both = test[test['HomePlanet'].isnull() & test['CabinDeck'].isnull()]
print(missing_both[['PassengerId']])

CabinDeck
F    16
E     9
G     9
D     5
B     3
C     2
A     1
Name: count, dtype: int64
     PassengerId
3421     7469_01


In [12]:
def fill_homeplanet(row):
    if pd.isnull(row['HomePlanet']) & pd.notnull(row['CabinDeck']):
        if row['CabinDeck'] in ['A', 'B', 'C', 'T']:
            return "Europa"
        elif row['CabinDeck'] in ['E', 'G']:
            return "Earth"
        elif row['CabinDeck'] in ['D', 'F']:
            return 'Mars'
    return row['HomePlanet']

test['HomePlanet'] = test.apply(fill_homeplanet, axis = 1) #do ta đưa từng hàng test vào mà 

most_common = test['HomePlanet'].mode()[0]
test['HomePlanet'] = test['HomePlanet'].fillna(most_common)

In [13]:
def fill_cabindeck(row):
    if pd.isnull(row['CabinDeck']):
        if row['HomePlanet'] == 'Europa':
            return 'A'
        elif row['HomePlanet'] == 'Earth':
            return 'E'
        elif row['HomePlanet'] == 'Mars':
            return 'D'
    return row['CabinDeck']

test['CabinDeck'] = test.apply(fill_cabindeck, axis = 1)

In [15]:
cat_drop = ['Cabin', 'Name', 'GroupId']
test = test.drop(columns=cat_drop, errors = 'ignore')

test['CryoSleep'] = test['CryoSleep'].fillna(test['CryoSleep'].mode()[0])
test['Destination'] = test['Destination'].fillna(test['Destination'].mode()[0])
test['VIP'] = test['VIP'].fillna(test['VIP'].mode()[0])

C:\Users\ADMIN\AppData\Local\Temp\ipykernel_8852\1739137414.py:4: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  test['CryoSleep'] = test['CryoSleep'].fillna(test['CryoSleep'].mode()[0])
C:\Users\ADMIN\AppData\Local\Temp\ipykernel_8852\1739137414.py:6: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  test['VIP'] = test['VIP'].fillna(test['VIP'].mode()[0])


In [18]:
numerical_test = test.select_dtypes(include=['int64', 'float64']).columns
categorical_test = test.select_dtypes(include='object').columns

print(numerical_test.tolist())
print(categorical_test.tolist())

['Age', 'RoomService', 'FoodCourt', 'ShoppingMall', 'Spa', 'VRDeck']
['PassengerId', 'HomePlanet', 'Destination', 'CabinDeck']


In [22]:
import joblib
import pandas as pd

# Load lại mô hình đã lưu
loaded_model = joblib.load('lightgbm_model.pkl')

numerical_features = ['Age', 'RoomService', 'FoodCourt', 'ShoppingMall', 'Spa', 'VRDeck']
categorical_features = ['Destination', 'CabinDeck']

# Giả sử bạn đã xử lý missing values cho test xong
X_test = test[numerical_features + categorical_features]

# Dự đoán
y_pred = loaded_model.predict(X_test)

# Convert sang True/False
y_pred_bool = y_pred.astype(bool)

# Nếu muốn xác suất
y_proba = loaded_model.predict_proba(X_test)[:, 1]

# In kết quả mẫu
print("Predictions:", y_pred_bool[:10])
print("Probabilities:", y_proba[:10])

# Tạo file submission (PassengerId + Transported)
submission = pd.DataFrame({
    "PassengerId": test["PassengerId"],   # giữ nguyên ID từ file test
    "Transported": y_pred_bool            # kết quả dự đoán dạng True/False
})

# Lưu ra CSV
submission.to_csv("submission.csv", index=False)

print("File submission.csv đã được tạo thành công!")


Predictions: [ True False  True  True False  True  True  True  True  True]
Probabilities: [6.08885378e-01 6.16828711e-04 9.97631409e-01 9.98294036e-01
 4.69459442e-01 5.76116204e-01 9.96561876e-01 9.20202437e-01
 9.77749122e-01 5.34857936e-01]
File submission.csv đã được tạo thành công!


c:\Users\ADMIN\Desktop\Cbruh\PPE-Detection\venv\lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
c:\Users\ADMIN\Desktop\Cbruh\PPE-Detection\venv\lib\site-packages\sklearn\utils\validation.py:2749: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
